# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaksimNikolin/FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb)

**How to run this notebook (Colab):** open it in Colab, add your Hugging Face READ token as a Secret named `HF_TOKEN` (key icon on the left, notebook access ON), then *Runtime → Run all*. The token is never printed or saved in the notebook. Expect a few minutes: it reads one month of the daily fact table (`month=2026-03`) over the network.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd

def get_token():
    try:
        from google.colab import userdata          # Colab Secrets panel
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face READ token: ")

# Default: the gated release on Hugging Face. WAREHOUSE_REL can point to a local mirror (see notebook 03).
REL = os.environ.get("WAREHOUSE_REL", "hf://datasets/FlyRank/internship-warehouse")
con = duckdb.connect()
if REL.startswith("hf://"):
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{get_token()}')")

MONTH = "2026-03"                                   # mid-panel development month; June 2026 stays sealed
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/**/*.parquet')"
print("Reading partition month =", MONTH)

## 1. Unit of analysis + time window

The contract for **Lane 4: CTR / Engagement Opportunity Scoring**, in plain words:

1. **What one row means.** In the source table, one row is **one content item on one day** (`report_date × client_hash_id × content_hash_id`). In my lane frame, one row is **one visible page over one month**, split into two windows.
2. **Which table.** `fact_content_daily_performance`, only the `month=2026-03` partition for development. Columns: `report_date`, `client_hash_id`, `content_hash_id`, `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_data_available`. `dim_clients` is used only to reason about history coverage.
3. **Which time window.** **Feature window:** Mar 1–15, 2026. **Target window:** Mar 16–31, 2026. Nothing from the target window may enter a feature. June 2026, the final month, stays a sealed test month and is not used here.
4. **What I rank (label / proxy).** `stays_under` = 1 if the page's CTR in Mar 16–31 is below **half** of its position tier's pooled CTR for the same days. The tier comes from the feature-window position. The queue ranks pages by how likely they are to keep under-capturing clicks.
5. **One thing I deliberately exclude.** Every **target-window** metric (Mar 16–31 impressions, clicks, CTR, position) as a feature. They are what the label is computed from. Also excluded: `client_hash_id` / `content_hash_id` as features (grouping and splitting only) and the query table `fact_content_query_90d`, whose fixed 90-day window would overlap the label window.

In [ ]:
CONTRACT = {
    "row (source)": "content item x day (report_date, client_hash_id, content_hash_id)",
    "row (lane frame)": "one visible page, March 2026, two half-month windows",
    "table": "fact_content_daily_performance, partition month=2026-03",
    "feature window": "2026-03-01 .. 2026-03-15",
    "target window": "2026-03-16 .. 2026-03-31",
    "label": "stays_under: CTR in target window < 0.5 x tier pooled CTR",
    "excluded": "target-window metrics as features; IDs as features; query_90d table",
}
for k, v in CONTRACT.items():
    print(f"{k:>17}: {v}")

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (Mar 1–15) | **Feature source** | Known on Mar 15, the decision moment |
| `gsc_impressions`, `gsc_clicks` (Mar 16–31) | **Label source** | The label is computed from them, so they are never features |
| `report_date` | **Context** | Splits rows into the feature and target windows |
| `client_hash_id`, `content_hash_id` | **Context** | Grouping, joins and the client-grouped split only |
| `ga4_data_available` | **Context** | Availability check; GA4 columns are zero-filled when FALSE |
| `gsc_avg_position` (Mar 16–31) | **Excluded** | Future information about the label window |
| `fact_content_query_90d` | **Excluded** | Its fixed 90-day window overlaps the label window |
| Product flags / scores | **Excluded** | Not in the release, by design (observable signals only) |

In [ ]:
FIELDS = pd.DataFrame([
    ("gsc_impressions / gsc_clicks / gsc_avg_position (Mar 1-15)", "feature source"),
    ("gsc_impressions / gsc_clicks (Mar 16-31)", "label source"),
    ("report_date", "context"),
    ("client_hash_id / content_hash_id", "context (grouping only)"),
    ("ga4_data_available", "context (availability)"),
    ("gsc_avg_position (Mar 16-31)", "excluded (future)"),
    ("fact_content_query_90d", "excluded (window overlap)"),
], columns=["field", "bucket"])
FIELDS

## 3. Verify it with queries (grain, counts, missing values, windows)

Exactly three verification queries on `month=2026-03`, then the five-feature frame and the deliberate leak.

**Query 1: grain.** Claim: one row = one content item per day. If the grain holds, no `(report_date, client, content)` combination appears twice.

In [ ]:
q1 = con.sql(f"""
    SELECT COUNT(*) AS duplicate_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {FACT_MONTH}
        GROUP BY 1, 2, 3
        HAVING c > 1
    )
""").df()
q1

**Query 2: slice row count and date span.** Claim: the partition covers Mar 1–31, 2026. My lane slice is the rows where the page was actually shown in search (`gsc_impressions > 0`).

In [ ]:
q2 = con.sql(f"""
    SELECT
        COUNT(*)                                                   AS all_rows,
        COUNT(*) FILTER (WHERE gsc_impressions > 0)                AS lane_rows_with_impressions,
        COUNT(DISTINCT content_hash_id) FILTER (WHERE gsc_impressions > 0) AS pages_with_impressions,
        COUNT(DISTINCT client_hash_id)  FILTER (WHERE gsc_impressions > 0) AS clients_with_impressions,
        MIN(report_date)                                           AS first_day,
        MAX(report_date)                                           AS last_day
    FROM {FACT_MONTH}
""").df()
q2.T.rename(columns={0: "value"})

**Query 3: availability.** Claim: not every row has GA4 behind it. Rows before a client's GA4 start carry zero-filled GA4 columns with `ga4_data_available = FALSE`. Filter with `IS TRUE`: a NULL flag is neither TRUE nor FALSE, and `IS TRUE` drops it too.

In [ ]:
q3 = con.sql(f"""
    SELECT
        COUNT(*)                                                         AS all_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)               AS rows_ga4_available,
        COUNT(*) FILTER (WHERE gsc_impressions > 0)                      AS lane_rows,
        COUNT(*) FILTER (WHERE gsc_impressions > 0 AND ga4_data_available IS TRUE) AS lane_rows_ga4_available
    FROM {FACT_MONTH}
""").df()
q3["share_ga4_available"] = (q3["rows_ga4_available"] / q3["all_rows"]).round(3)
q3["lane_share_ga4_available"] = (q3["lane_rows_ga4_available"] / q3["lane_rows"]).round(3)
q3.T.rename(columns={0: "value"})

### Five features, max, and when each one is knowable

The decision moment is **Mar 15**: an editor plans the second half of the month. Everything below is computed only from **Mar 1–15**.

| Feature | Knowable at the decision moment because… |
|---|---|
| `imp_w1`: impressions, Mar 1–15 | It is the sum of days that already happened by Mar 15. |
| `ctr_w1`: CTR (%), Mar 1–15 | Clicks and impressions from the same finished days. |
| `pos_w1`: impression-weighted average position, Mar 1–15 | Search Console reports position daily; Mar 1–15 is complete on Mar 15. |
| `ctr_gap_w1`: `ctr_w1` ÷ pooled CTR of the page's position tier, Mar 1–15 | The tier benchmark uses only Mar 1–15 data from all pages. |
| `days_visible_w1`: days with impressions, Mar 1–15 | It counts past days only, a stability signal. |

Eligibility, a policy choice: positions 1–20 in the feature window, and at least 100 impressions in **each** half, so the CTR is not built on a handful of views.

In [ ]:
frame = con.sql(f"""
    WITH d AS (
        SELECT client_hash_id, content_hash_id, report_date,
               COALESCE(gsc_impressions, 0) AS imp,
               COALESCE(gsc_clicks, 0)      AS clk,
               gsc_avg_position             AS pos
        FROM {FACT_MONTH}
        WHERE gsc_impressions > 0
    )
    SELECT client_hash_id, content_hash_id,
           SUM(imp) FILTER (WHERE report_date <= DATE '2026-03-15')                 AS imp_w1,
           SUM(clk) FILTER (WHERE report_date <= DATE '2026-03-15')                 AS clk_w1,
           SUM(pos * imp) FILTER (WHERE report_date <= DATE '2026-03-15' AND pos > 0)
             / NULLIF(SUM(imp) FILTER (WHERE report_date <= DATE '2026-03-15' AND pos > 0), 0) AS pos_w1,
           COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '2026-03-15') AS days_visible_w1,
           SUM(imp) FILTER (WHERE report_date >  DATE '2026-03-15')                 AS imp_w2,
           SUM(clk) FILTER (WHERE report_date >  DATE '2026-03-15')                 AS clk_w2
    FROM d
    GROUP BY 1, 2
""").df()

MIN_IMP = 100
f = frame[(frame["imp_w1"] >= MIN_IMP) & (frame["imp_w2"] >= MIN_IMP) & (frame["pos_w1"].between(1, 20))].copy()
f["tier"] = pd.cut(f["pos_w1"], bins=[0, 3, 10, 20], labels=["top_3", "page_1", "striking"])
f["ctr_w1"] = 100 * f["clk_w1"] / f["imp_w1"]
f["ctr_w2"] = 100 * f["clk_w2"] / f["imp_w2"]

tier_w1 = f.groupby("tier", observed=True)[["clk_w1", "imp_w1"]].sum()
tier_w2 = f.groupby("tier", observed=True)[["clk_w2", "imp_w2"]].sum()
f["tier_ctr_w1"] = f["tier"].map(100 * tier_w1["clk_w1"] / tier_w1["imp_w1"]).astype(float)
f["tier_ctr_w2"] = f["tier"].map(100 * tier_w2["clk_w2"] / tier_w2["imp_w2"]).astype(float)
f["ctr_gap_w1"] = f["ctr_w1"] / f["tier_ctr_w1"]

# LABEL (target window only)
f["stays_under"] = (f["ctr_w2"] < 0.5 * f["tier_ctr_w2"]).astype(int)

FEATURES = ["imp_w1", "ctr_w1", "pos_w1", "ctr_gap_w1", "days_visible_w1"]
print(f"Lane frame: {len(f):,} pages from {f['client_hash_id'].nunique()} clients | label base rate {f['stays_under'].mean():.1%}")
f[FEATURES + ["stays_under"]].describe().T.round(3)

### The trap, performed on purpose

A quick honest score first: logistic regression on the five features, tested on **clients the model never saw** (`GroupShuffleSplit` on `client_hash_id`). Metrics: ROC-AUC, and precision@50, since the output is a review queue.

Then I add **one label-derived column**, `ctr_w2_vs_tier` (second-half CTR ÷ second-half tier CTR). The label is literally `ctr_w2_vs_tier < 0.5`. Watch the score jump toward perfect. Then I delete it and keep the honest number.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

f["log_imp_w1"] = np.log1p(f["imp_w1"])
def quick_score(frame, cols):
    X, y, g = frame[cols].fillna(0), frame["stays_under"], frame["client_hash_id"]
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(X, y, g))
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X.iloc[tr], y.iloc[tr])
    p = model.predict_proba(X.iloc[te])[:, 1]
    top50 = y.iloc[te].to_numpy()[np.argsort(-p)[:50]]
    return roc_auc_score(y.iloc[te], p), top50.mean(), y.iloc[te].mean()

HONEST = ["log_imp_w1", "ctr_w1", "pos_w1", "ctr_gap_w1", "days_visible_w1"]   # the 5 features (impressions log-scaled)
auc, p50, base = quick_score(f, HONEST)
print(f"HONEST  | ROC-AUC {auc:.3f} | precision@50 {p50:.1%} | base rate in test {base:.1%}")

# --- the deliberate leak ---
f["ctr_w2_vs_tier"] = f["ctr_w2"] / f["tier_ctr_w2"]          # computed from the TARGET window
auc_l, p50_l, _ = quick_score(f, HONEST + ["ctr_w2_vs_tier"])
print(f"LEAKED  | ROC-AUC {auc_l:.3f} | precision@50 {p50_l:.1%}   <- too good to be true: the column IS the label")

# --- remove it and keep the honest number ---
f = f.drop(columns=["ctr_w2_vs_tier"])
assert "ctr_w2_vs_tier" not in f.columns
auc, p50, base = quick_score(f, HONEST)
print(f"KEPT    | ROC-AUC {auc:.3f} | precision@50 {p50:.1%}  (honest number, leak column deleted)")

## 4. Data limits

**The named limitation of my slice: half-month windows on an unbalanced panel.** I build features and the label from two 15–16-day windows of one month. That keeps it fast and leakage-safe, but:

- Short windows make CTR noisy for mid-volume pages, even with the 100-impression floor. Some `stays_under = 1` labels will be one-off wobbles.
- Clients entered tracking at different dates (`dim_clients.gsc_data_start`). A single month hides that some pages are brand new to Search Console tracking while others have a year of history.
- Position is an impression-weighted average of daily averages. It blurs pages that moved between tiers during the window.

Also true, and never claimable from this data: why a page's CTR is low (SERP features, AI answers, brand effects aren't in the table), or that a title rewrite would fix it. The output stays *observed, directional, decision-support*.

In [ ]:
LIMITATION = ("Half-month windows on an unbalanced panel: CTR is noisy over 15 days, "
              "client histories start at different dates, and averaged position blurs tier moves.")
print(LIMITATION)
print(f"Pages near the volume floor (100-300 impressions in the feature window): "
      f"{((f['imp_w1'] >= 100) & (f['imp_w1'] < 300)).mean():.1%} of the lane frame")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.